# Домашнее задание №1: Tabular Supervised Classification

## Постановка задачи
Необходимо обучить модель машинного обучения (нейронную сеть прямого распространения - FFN) на табличных данных `train.csv` и сгенерировать предсказания для тестовой выборки `test_features.csv`.
Целевая переменная — `prediction` (бинарная классификация: 0 или 1).
Результат должен быть сохранен в файл `outputs/task1_predictions.csv` с колонками `patient_id` и `prediction`.

## Декомпозиция на шаги:
1. **Загрузка данных:** Считываем обучающую и тестовую выборки.
2. **Предобработка данных:** 
   - Обрабатываем пропуски (заполняем медианой, рассчитанной по обучающей выборке).
   - Масштабируем признаки (StandardScaler), так как нейросети чувствительны к масштабу данных.
3. **Разделение данных:** Отделяем часть обучающей выборки для валидации, чтобы контролировать процесс обучения.
4. **Определение модели:** Создаем класс FFN (Feed-Forward Network) на базе PyTorch.
5. **Обучение:** Запускаем цикл обучения с использованием оптимизатора Adam и функции потерь BCEWithLogitsLoss.
6. **Оценка:** Считаем метрики качества на валидационной выборке.
7. **Предсказание и сохранение:** Делаем прогноз для тестовой выборки и сохраняем результат в CSV-файл.

In [1]:
# импортируб библиотеки
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset

## Шаг 1: Загрузка данных
Считываем файлы `train.csv` и `test_features.csv`. 
*Важно:* предполагается, что ноутбук запускается из корневой папки репозитория, поэтому пути к файлам относительные!!

In [2]:
# Загружаем данные
train_df = pd.read_csv("../dataset/task1/train.csv")
test_df = pd.read_csv("../dataset/task1/test_features.csv")

# Определяем признаки (все колонки, кроме patient_id и целевой переменной)
feature_cols = [c for c in test_df.columns if c != "patient_id"]

# Формируем матрицы признаков и целевую переменную
X_train_df = train_df[feature_cols].copy()
y_train = train_df["prediction"].values.astype(np.float32).reshape(-1, 1)
X_test_df = test_df[feature_cols].copy()

## Шаг 2: Предобработка данных
1. **Пропуски:** В данных встречаются пропуски (например, в ЛПНП или СКФ). Заполним их медианой, рассчитанной *только по обучающей выборке*, чтобы избежать утечки данных (data leakage).
2. **Масштабирование:** Нейронные сети работают лучше, когда все признаки находятся в одном масштабе. Используем `StandardScaler`.

In [3]:
# Заполняем пропуски 
X_train_df = X_train_df.fillna(X_train_df.median())
X_test_df = X_test_df.fillna(X_train_df.median()) 

# Масштабируем признаки
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train_df)
X_test_scaled = scaler.transform(X_test_df)

## Шаг 3: Разделение на Train/Validation
Разделим обучающую выборку на две части: 80% для обучения и 20% для валидации. 
Валидация нужна, чтобы проверять, не переобучается ли модель, и вовремя остановиться, если качество на новых данных падает.

In [4]:
# Разделяем train на обучающую и валидационную выборки
X_tr, X_val, y_tr, y_val = train_test_split(
    X_train_scaled, y_train, test_size=0.2, random_state=42, stratify=y_train
)

# Преобразуем данные в тензоры PyTorch
X_tr_t = torch.tensor(X_tr, dtype=torch.float32)
y_tr_t = torch.tensor(y_tr, dtype=torch.float32)
X_val_t = torch.tensor(X_val, dtype=torch.float32)
y_val_t = torch.tensor(y_val, dtype=torch.float32)

# Создаем DataLoader для подачи данных батчами
batch_size = 32
train_loader = DataLoader(TensorDataset(X_tr_t, y_tr_t), batch_size=batch_size, shuffle=True)
val_loader = DataLoader(TensorDataset(X_val_t, y_val_t), batch_size=batch_size, shuffle=False)

## Шаг 4: Определение модели FFN
Создаем полносвязную нейронную сеть (FFN) по аналогии с примером:
- `fc1`: Входной слой (принимает количество признаков, выдает 64 нейрона).
- `fc2`: Скрытый слой (64 -> 32 нейрона).
- `fc3`: Выходной слой (32 -> 1 нейрон, так как бинарная классификация).
- `ReLU`: Функция активации, добавляет нелинейность.
- `Dropout`: Случайно отключает 20% нейронов во время обучения, чтобы бороться с переобучением.

In [5]:
# Определяем класс нейронной сети
class TabularFFN(nn.Module):
    def __init__(self, input_dim):
        super().__init__()
        # Первый линейный слой
        self.fc1 = nn.Linear(input_dim, 64)
        # Второй линейный слой
        self.fc2 = nn.Linear(64, 32)
        # Выходной линейный слой (один нейрон для бинарной классификации)
        self.fc3 = nn.Linear(32, 1)
        # Функция активации ReLU
        self.relu = nn.ReLU()
        # Слой Dropout для регуляризации
        self.dropout = nn.Dropout(0.2)

    def forward(self, x):
        # Пропускаем данные через слои с активацией и dropout
        x = self.relu(self.fc1(x))
        x = self.dropout(x)
        x = self.relu(self.fc2(x))
        x = self.fc3(x) # Выход без активации (логиты)
        return x

# Определяем устройство (GPU или CPU)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
# Создаем экземпляр модели
model = TabularFFN(X_train_scaled.shape[1]).to(device)

## Шаг 5: Обучение модели
- **Функция потерь:** `BCEWithLogitsLoss` — стандарт для бинарной классификации (объединяет Sigmoid и BCELoss).
- **Оптимизатор:** `Adam` с шагом обучения `lr=0.001`.
- **Цикл обучения:** Проходим по эпохам, делаем forward pass, считаем ошибку, делаем backpropagation и обновляем веса.

In [6]:
# Функция потерь и оптимизатор
criterion = nn.BCEWithLogitsLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

# Параметры обучения
epochs = 100
train_losses, val_losses = [], []

for epoch in range(epochs):
    # --- Режим обучения ---
    model.train()
    running_loss = 0.0
    for X_batch, y_batch in train_loader:
        X_batch, y_batch = X_batch.to(device), y_batch.to(device)
        
        optimizer.zero_grad() # Обнуляем градиенты
        outputs = model(X_batch) # Прямой проход
        loss = criterion(outputs, y_batch) # Считаем ошибку
        loss.backward() # Обратный проход (градиенты)
        optimizer.step() # Шаг оптимизатора
        
        running_loss += loss.item() * X_batch.size(0)
    
    epoch_train_loss = running_loss / len(train_loader.dataset)
    train_losses.append(epoch_train_loss)

    # --- Режим валидации ---
    model.eval()
    val_loss = 0.0
    with torch.no_grad(): # Отключаем расчет градиентов
        for X_batch, y_batch in val_loader:
            X_batch, y_batch = X_batch.to(device), y_batch.to(device)
            outputs = model(X_batch)
            loss = criterion(outputs, y_batch)
            val_loss += loss.item() * X_batch.size(0)
    
    epoch_val_loss = val_loss / len(val_loader.dataset)
    val_losses.append(epoch_val_loss)

    # Выводим прогресс каждые 20 эпох
    if (epoch + 1) % 20 == 0:
        print(f"Epoch {epoch+1}/{epochs} — Train Loss: {epoch_train_loss:.4f}, Val Loss: {epoch_val_loss:.4f}")

Epoch 20/100 — Train Loss: 0.0891, Val Loss: 0.1526
Epoch 40/100 — Train Loss: 0.0447, Val Loss: 0.1662
Epoch 60/100 — Train Loss: 0.0328, Val Loss: 0.1913
Epoch 80/100 — Train Loss: 0.0073, Val Loss: 0.2389
Epoch 100/100 — Train Loss: 0.0081, Val Loss: 0.2580


## Шаг 6: Оценка качества модели
После обучения проверим метрики на валидационной выборке. 
Так как данные могут быть несбалансированы, смотрим не только на Accuracy, но и на F1-score и ROC-AUC.

In [7]:
model.eval()
val_probs = []
val_true = []

with torch.no_grad():
    for X_batch, y_batch in val_loader:
        X_batch = X_batch.to(device)
        outputs = model(X_batch)
        probs = torch.sigmoid(outputs).cpu().numpy().flatten()
        val_probs.extend(probs)
        val_true.extend(y_batch.numpy().flatten())

# Переводим вероятности в классы (порог 0.5)
val_pred = (np.array(val_probs) >= 0.5).astype(int)

print("="*50)
print("Оценка на валидационной выборке:")
print("="*50)
print(f"Accuracy:  {accuracy_score(val_true, val_pred):.4f}")
print(f"F1-score:  {f1_score(val_true, val_pred):.4f}")
print(f"ROC-AUC:   {roc_auc_score(val_true, val_probs):.4f}")

Оценка на валидационной выборке:
Accuracy:  0.9375
F1-score:  0.8077
ROC-AUC:   0.9800


## Шаг 7: Предсказание для тестовой выборки и сохранение
1. Преобразуем тестовые данные в тензор.
2. Делаем предсказание (получаем вероятности).
3. Переводим вероятности в классы 0 или 1 с порогом 0.5.
4. Формируем DataFrame с `patient_id` и `prediction`.
5. Сохраняем в `outputs/task1_predictions.csv` (папка `outputs` уже есть в репозитории).

In [9]:
# Преобразуем тестовые данные в тензор
X_test_t = torch.tensor(X_test_scaled, dtype=torch.float32)

# Делаем предсказание
model.eval()
with torch.no_grad():
    logits = model(X_test_t.to(device))
    test_probs = torch.sigmoid(logits).cpu().numpy().flatten()

# Превращаем вероятности в классы 0 или 1
test_predictions = (test_probs >= 0.5).astype(int)

# Сохранение результата
result_df = pd.DataFrame({
    "patient_id": test_df["patient_id"],
    "prediction": test_predictions
})

# Сохраняем в CSV
result_df.to_csv("../outputs/task1_predictions.csv", index=False)

print(f"\nФайл успешно сохранен: outputs/task1_predictions.csv")
print(result_df.head())


Файл успешно сохранен: outputs/task1_predictions.csv
   patient_id  prediction
0           2           0
1           3           0
2           8           0
3          18           0
4          19           0
